# 05 — Retrieval Evaluation

Compare the three production retrievers on the annotated benchmark:

- Tantivy BM25
- FAISS embeddings
- Hybrid retrieval

Metric implementations live in `src/rag/evaluation/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.evaluation import (
    EvaluationDataset,
    RetrievalEvaluator,
    compare_retrievers,
    bootstrap_confidence_intervals,
)
from src.rag.runtime import load_retrieval_stack

In [ ]:
EVAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "retrieval_queries.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

KS = (1, 3, 5, 10)

dataset = EvaluationDataset(EVAL_PATH)
stack = load_retrieval_stack(PROJECT_ROOT)

retrievers = {
    "bm25": stack.bm25,
    "embedding": stack.embedding,
    "hybrid": stack.hybrid,
}

print("Samples:", len(dataset))
print(
    "Products:",
    len({sample["product_id"] for sample in dataset}),
)

## Smoke test

In [ ]:
sample = dataset.samples[0]

print("Query:", sample["query"])
print("Relevant IDs:", sample["relevant_ids"])

for name, retriever in retrievers.items():
    print("\n", name.upper())

    results = retriever.retrieve(
        sample["query"],
        top_k=5,
        candidate_ids=sample["candidate_ids"],
    )

    display(results[["id", "score", "body"]])

## Full benchmark

In [ ]:
summary, per_query = compare_retrievers(
    retrievers=retrievers,
    dataset=dataset,
    ks=KS,
)

headline_columns = [
    "precision@5",
    "recall@5",
    "hit_rate@5",
    "mrr@5",
    "map@5",
    "ndcg@5",
    "latency_mean_ms",
    "latency_p50_ms",
    "latency_p95_ms",
]

display(summary.round(4))
display(summary[headline_columns].round(4))

## Quality comparison

In [ ]:
for metric in [
    "recall@5",
    "mrr@5",
    "ndcg@5",
    "hit_rate@5",
]:
    ax = summary[metric].plot(
        kind="bar",
        figsize=(7, 4),
        title=metric,
    )
    ax.set_xlabel("Retriever")
    ax.set_ylabel(metric)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Confidence intervals

In [ ]:
confidence_intervals = bootstrap_confidence_intervals(
    per_query,
    metrics=("recall@5", "mrr@5", "ndcg@5"),
)

display(confidence_intervals.round(4))

## Failure analysis

In [ ]:
for name in retrievers:
    method_results = per_query[
        per_query["retriever"] == name
    ]

    print("\n", name.upper())

    display(
        RetrievalEvaluator.failure_cases(
            method_results,
            metric="recall@5",
            n=10,
        )[
            [
                "product_id",
                "query",
                "relevant_ids",
                "retrieved_ids",
                "recall@5",
                "latency_ms",
            ]
        ]
    )

## Save artifacts

In [ ]:
summary.to_csv(
    OUTPUT_DIR / "retrieval_summary.csv",
    encoding="utf-8-sig",
)

per_query.to_json(
    OUTPUT_DIR / "retrieval_per_query.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

confidence_intervals.to_csv(
    OUTPUT_DIR / "retrieval_confidence_intervals.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", OUTPUT_DIR)